---

Since most datasets are in the form of YOLO already, this will be a quick check section, rather than real scaling. Even though real rescaling will be needed when images from different sources are passed through

---

In [8]:
import os
import shutil
import yaml
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

import cv2
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT       = Path(r"D:\SIT374\WalkBuddy-T2-2026\ML_side")
CONFIG_DIR         = PROJECT_ROOT / "config"
DATASETS_DIR       = PROJECT_ROOT / "datasets"

INTERIM_MAPPED_DIR = DATASETS_DIR / "interim_mapped"
REPORTS_DIR        = DATASETS_DIR / "reports"

TAXONOMY_YAML      = CONFIG_DIR / "taxonomy_mapping.yaml"
DOWNLOAD_YAML      = CONFIG_DIR / "download_sheet.yaml"

CLASS_COUNT_CSV    = REPORTS_DIR / "per_class_instance_counts.csv"
INVENTORY_CSV      = REPORTS_DIR / "dataset_inventory.csv"
EXCLUSIONS_JSON    = REPORTS_DIR / "exclusions.json"

TAXONOMY = ["person", "stairs", "door", "chair", "table", "pole", "bicycle", "vehicle"]
CLASS_ID_TO_NAME = {i: name for i, name in enumerate(TAXONOMY)}

assert INTERIM_MAPPED_DIR.exists(), f"INTERIM_MAPPED_DIR not found: {INTERIM_MAPPED_DIR}"
assert TAXONOMY_YAML.exists(), f"taxonomy_mapping.yaml not found: {TAXONOMY_YAML}"

with open(TAXONOMY_YAML, "r") as f:
    taxonomy_cfg = yaml.safe_load(f)

print(f"INTERIM_MAPPED_DIR: {INTERIM_MAPPED_DIR}")
print(f"Taxonomy: {TAXONOMY}")

INTERIM_MAPPED_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim_mapped
Taxonomy: ['person', 'stairs', 'door', 'chair', 'table', 'pole', 'bicycle', 'vehicle']


In [2]:
class_counts = {
    "person": 25714, "stairs": 3414, "door": 21839, "chair": 5072,
    "table": 8608, "pole": 5665, "bicycle": 5307, "vehicle": 16407,
}

import csv
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
with open(CLASS_COUNT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class", "instance_count"])
    for cls in TAXONOMY:
        writer.writerow([cls, class_counts[cls]])

print(f"Saved to {CLASS_COUNT_CSV}")

Saved to D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\per_class_instance_counts.csv


In [24]:
def verify_pair(img_path, lbl_path):
    issues = []
    warnings = []

    # Verify image
    try:
        with Image.open(img_path) as im:
            im.verify()

        with Image.open(img_path) as im:
            w, h = im.size

    except Exception as e:
        return {
            "image": str(img_path),
            "label": str(lbl_path),
            "issues": [f"unreadable_image: {e}"],
            "warnings": []
        }

    if w < 10 or h < 10:
        issues.append(f"suspicious_dims_{w}x{h}")

    # Check label
    if not lbl_path.exists():
        issues.append("missing_label_file")
        return {
            "image": str(img_path),
            "label": str(lbl_path),
            "issues": issues,
            "warnings": warnings
        }

    try:
        with open(lbl_path, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
    except Exception as e:
        issues.append(f"unreadable_label: {e}")
        return {
            "image": str(img_path),
            "label": str(lbl_path),
            "issues": issues,
            "warnings": warnings
        }

    for i, line in enumerate(lines):

        parts = line.split()

        # Class ID
        try:
            cls_id = int(parts[0])
        except (ValueError, IndexError):
            issues.append(f"line{i}_non_numeric_class_id")
            continue

        if cls_id not in CLASS_ID_TO_NAME:
            issues.append(f"line{i}_bad_class_id_{cls_id}")

        coordinates = parts[1:]

        # YOLO detection: class x_center y_center width height
        if len(parts) == 5:

            try:
                x, y, bw, bh = map(float, coordinates)
            except ValueError:
                issues.append(f"line{i}_non_numeric")
                continue

            # Record boundary overflow as a warning
            if not (0.0 <= x <= 1.0):
                warnings.append(
                    f"line{i}_x_out_of_range_{x:.4f}"
                )

            if not (0.0 <= y <= 1.0):
                warnings.append(
                    f"line{i}_y_out_of_range_{y:.4f}"
                )

            if not (0.0 <= bw <= 1.0):
                warnings.append(
                    f"line{i}_bw_out_of_range_{bw:.4f}"
                )

            if not (0.0 <= bh <= 1.0):
                warnings.append(
                    f"line{i}_bh_out_of_range_{bh:.4f}"
                )

            if bw <= 0 or bh <= 0:
                issues.append(f"line{i}_zero_or_negative_box")
                continue

            # Calculate bounding-box boundaries
            x1 = x - bw / 2
            y1 = y - bh / 2
            x2 = x + bw / 2
            y2 = y + bh / 2

            # Actual box must overlap the image
            if x2 <= 0 or x1 >= 1:
                issues.append(
                    f"line{i}_box_outside_horizontal"
                )

            if y2 <= 0 or y1 >= 1:
                issues.append(
                    f"line{i}_box_outside_vertical"
                )

        # YOLO segmentation: class x1 y1 x2 y2 ...
        elif len(parts) >= 7 and len(coordinates) % 2 == 0:

            try:
                coords = list(map(float, coordinates))
            except ValueError:
                issues.append(f"line{i}_non_numeric")
                continue

            if len(coords) < 6:
                issues.append(
                    f"line{i}_insufficient_polygon_points"
                )
                continue

            # Boundary overflow is a warning
            for j, val in enumerate(coords):

                coord_type = "x" if j % 2 == 0 else "y"

                if not (0.0 <= val <= 1.0):
                    warnings.append(
                        f"line{i}_{coord_type}_out_of_range_{val:.4f}"
                    )

            xs = coords[0::2]
            ys = coords[1::2]

            # Polygon must overlap the image
            if max(xs) <= 0 or min(xs) >= 1:
                issues.append(
                    f"line{i}_polygon_outside_horizontal"
                )

            if max(ys) <= 0 or min(ys) >= 1:
                issues.append(
                    f"line{i}_polygon_outside_vertical"
                )

        else:
            issues.append(
                f"line{i}_bad_field_count_{len(parts)}"
            )

    return {
        "image": str(img_path),
        "label": str(lbl_path),
        "issues": issues,
        "warnings": warnings
    }


def run_verification(interim_mapped_dir):

    image_exts = {".jpg", ".jpeg", ".png"}

    all_images = [
        p for p in interim_mapped_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    verification_issues = []
    boundary_warnings = []

    for img_path in tqdm(all_images, desc="verifying"):

        lbl_path = img_path.with_suffix(".txt")

        if not lbl_path.exists():
            labels_dir = img_path.parent.parent / "labels"
            candidate = labels_dir / f"{img_path.stem}.txt"

            if candidate.exists():
                lbl_path = candidate

        result = verify_pair(img_path, lbl_path)

        if result["issues"]:
            verification_issues.append(result)

        if result["warnings"]:
            boundary_warnings.append(result)

    return (
        verification_issues,
        boundary_warnings,
        len(all_images)
    )


# Run verification
verification_issues, boundary_warnings, total_checked = run_verification(
    INTERIM_MAPPED_DIR
)

print(f"Checked: {total_checked}")
print(f"Pairs with actual issues: {len(verification_issues)}")
print(f"Pairs with boundary warnings: {len(boundary_warnings)}")


# Summarise actual issues
if verification_issues:

    issue_types = Counter()

    for result in verification_issues:
        for issue in result["issues"]:
            issue_type = "_".join(issue.split("_")[:2])
            issue_types[issue_type] += 1

    print("\nIssue Summary:")

    for issue_type, count in issue_types.most_common():
        print(f"  {issue_type}: {count}")

else:
    print("No verification issues found.")


# Summarise boundary warnings
if boundary_warnings:

    warning_types = Counter()

    for result in boundary_warnings:
        for warning in result["warnings"]:
            warning_type = "_".join(warning.split("_")[:2])
            warning_types[warning_type] += 1

    print("\nBoundary Warning Summary:")

    for warning_type, count in warning_types.most_common():
        print(f"  {warning_type}: {count}")

else:
    print("No boundary warnings found.")

verifying:  34%|█████████████████████▋                                          | 12058/35633 [00:12<00:28, 820.51it/s]C:\Users\huynh\anaconda3\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (95664000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
verifying: 100%|████████████████████████████████████████████████████████████████| 35633/35633 [00:37<00:00, 949.90it/s]

Checked: 35633
Pairs with actual issues: 0
Pairs with boundary warnings: 41
No verification issues found.

Boundary Warning Summary:
  line0_bh: 36
  line0_y: 12
  line0_x: 11
  line0_bw: 1


---

Export a report

---

In [25]:
VERIFICATION_CSV = REPORTS_DIR / "scaling_verification_results.csv"

with open(VERIFICATION_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "label", "issues"])
    for r in verification_issues:
        writer.writerow([r["image"], r["label"], "; ".join(r["issues"])])

print(f"Saved: {VERIFICATION_CSV}")
print("Clean — nothing needs scaling" if not verification_issues else f"{len(verification_issues)} pairs flagged, review before proceeding")

Saved: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\scaling_verification_results.csv
Clean — nothing needs scaling


---

End of label scaling

---